# 04 — Simulação e validação do Erlang

As fórmulas de Erlang são **aproximações com hipóteses**. O simulador reproduz uma central chamada a chamada, com chegadas ao acaso, tempos de atendimento variáveis e clientes que desistem. Aqui você compara os dois e testa um plano de equipe.

Este notebook **não precisa do modelo TimesFM**.

## 1. Uma central em regime constante: fórmula × simulação

12 agentes, 10 Erlangs de carga, TMA de 300 s, meta de atender em até 20 s. Simulamos 40 intervalos de 30 minutos, 20 vezes, e descartamos os 4 primeiros (a central começa vazia).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from callcenter_lab.wfm import asa, service_level, simulate_replications

agents, load, aht, interval = 12, 10.0, 300.0, 30
calls = np.full(40, load * interval * 60 / aht)  # chamadas esperadas por intervalo
result = simulate_replications(calls, agents, aht, interval, n_replications=20, seed=1, warmup_intervals=4)

sl, sl_ci = result.overall['service_level_pct']
wait, wait_ci = result.overall['avg_wait_answered_seconds']
print(f'Fórmula (Erlang C): nível de serviço {100 * service_level(agents, load, aht, 20):.1f}%  espera {asa(agents, load, aht):.1f} s')
print(f'Simulação:          nível de serviço {sl:.1f}% ± {sl_ci:.1f}  espera {wait:.1f} s ± {wait_ci:.1f}')

O `±` é a margem de erro de 95% entre as 20 execuções. Quando a fórmula cai dentro dela, os dois concordam.

## 2. Quando os clientes desistem (Erlang A)

Mesma carga, mas só 9 agentes e clientes que desistem após 60 s em média. O Erlang C não enxerga o abandono; o Erlang A sim.

In [ ]:
from callcenter_lab.wfm import erlang_a

model = erlang_a(9, load, aht, patience_seconds=60, threshold_seconds=20)
sim = simulate_replications(calls, 9, aht, interval, n_replications=20, seed=2, warmup_intervals=4,
                            mean_patience_seconds=60)
print(f"Erlang A:   nível de serviço {100 * model['service_level']:.1f}%  abandono {100 * model['abandon_prob']:.1f}%")
sl_a, sl_a_ci = sim.overall['service_level_pct']
ab_a, ab_a_ci = sim.overall['abandon_pct']
print(f'Simulação:  nível de serviço {sl_a:.1f}% ± {sl_a_ci:.1f}  abandono {ab_a:.1f}% ± {ab_a_ci:.1f}')

## 3. Testando um plano de equipe de verdade

Dimensionamos um dia da central de exemplo e conferimos no simulador se a meta de 80% em 20 s é cumprida.

In [ ]:
from pathlib import Path

from callcenter_lab.loading import detect_columns, prepare_series, resample_series
from callcenter_lab.wfm import StaffingConfig, build_plan

df = pd.read_csv(Path('..') / 'data' / 'samples' / 'chamadas_sinteticas.csv')
series = resample_series(prepare_series(df, *detect_columns(df)), '30min', how='sum', fill='zero')
day = series.loc['2024-05-17']  # uma sexta-feira
config = StaffingConfig(service_level_pct=80, service_level_seconds=20)
plan = build_plan(day, config)
plan['agents'].max()

In [ ]:
check = simulate_replications(plan['calls'].to_numpy(), plan['agents'].to_numpy(), plan['aht_seconds'].to_numpy(),
                              30, n_replications=20, seed=3)
mean, ci = check.overall['service_level_pct']
print(f'Nível de serviço simulado do dia: {mean:.1f}% ± {ci:.1f} (meta {config.service_level_pct:.0f}%)')

In [ ]:
sim = check.intervals
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plan.index, plan['service_level_pct'], color='gray', label='Esperado (fórmula)')
ax.errorbar(plan.index, sim['service_level_pct'], yerr=sim['service_level_pct_ci'].fillna(0), color='tab:blue', label='Simulado (média ± IC)')
ax.axhline(config.service_level_pct, color='tab:orange', linestyle='--', label='Meta')
ax.set_ylabel('Nível de serviço (%)')
ax.legend()
plt.show()

Intervalos individuais oscilam bastante (poucas chamadas por intervalo), mas o dia todo deve ficar perto da meta.

## 4. E se faltarem agentes?

Tiramos 2 agentes de cada intervalo com atendimento e vemos o estrago.

In [ ]:
fewer = np.maximum(plan['agents'].to_numpy() - 2, 0)
worse = simulate_replications(plan['calls'].to_numpy(), fewer, plan['aht_seconds'].to_numpy(),
                              30, n_replications=20, seed=3)
worse_sl, worse_ci = worse.overall['service_level_pct']
print(f'Nível de serviço com 2 agentes a menos: {worse_sl:.1f}% ± {worse_ci:.1f}')

## Para experimentar

- Troque `n_replications` por 5 e por 50: a margem de erro muda?
- Rode a seção 3 com uma segunda-feira: `series.loc['2024-05-20']`.
- Use `mean_patience_seconds=120` na seção 3 e veja o abandono do dia.